# CS 521 HW3, Problem 1 (Adversarial Training).
In this question, you are asked to implement adversarial training algorithms with PGD.

(a) (25 points) Using the code you had written for HW1, perform (PGD-based) adversarial training. Use the starter notebook of Problem 3 as your code base: https:
//github.com/nalinwadhwa02/CCS521FA26HW/blob/main/hw3/q3/multi_
norm_adversarial_training_cifar10.ipynb. Train your models from
scratch: start from a new model (make model()), not from the base model of
Problem 3. Train two models for 5 epochs: a standard model and an ℓ∞ adversarially trained model. Report the standard accuracy, robust accuracy, and any
other observations with different epsilon values by filling in the blanks of the
provided code.

(b) (10 points) Try to attack the model you obtained after adversarial training using
PGD and compare the effectiveness (based on the lowering of accuracy) with that
of the original model. How effective is the model against an untargeted FGSM
attack (one step of PGD)? Record your observations and try to explain them.
Try different epsilon values against the adversarially trained models and report
your observations.
Hints. Adversarial training is nothing but adding adversarial examples during training. You can generate the adversarial examples in the designated space provided in the
notebook. The attack is the PGD attack you implemented in HW1.

Use Google Colab with a GPU: **Runtime > Change runtime type > T4 GPU**.

# Set up for dataset and model

Download the base model, a subset of 10,000 CIFAR-10 training images (1,000 for each class),
and the first 1,000 CIFAR-10 test images from Google Drive.

In [32]:
!gdown 1LoY_o51zNQkrkfvAlGE-44g9G-c38OJe -O cifar10_train_10k.pt
!gdown 1bElxIZ6A708IOGaTMcDq_AsCpC9QQseA -O cifar10_test_1k.pt

Downloading...
From (original): https://drive.google.com/uc?id=1LoY_o51zNQkrkfvAlGE-44g9G-c38OJe
From (redirected): https://drive.google.com/uc?id=1LoY_o51zNQkrkfvAlGE-44g9G-c38OJe&confirm=t&uuid=2ef458b7-2b2b-431e-b303-90137a70cf91
To: /content/cifar10_train_10k.pt
100% 30.8M/30.8M [00:00<00:00, 127MB/s]
Downloading...
From: https://drive.google.com/uc?id=1bElxIZ6A708IOGaTMcDq_AsCpC9QQseA
To: /content/cifar10_test_1k.pt
100% 3.08M/3.08M [00:00<00:00, 146MB/s]


In [33]:
import time

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

NORMS = ("linf", "l2", "l1")
EPS = {"linf": 8 / 255, "l2": 0.5, "l1": 12.0}   # used for training and test_model

EPS_SWEEP = {
    "linf": [1 / 255, 2 / 255, 4 / 255, 8 / 255, 16 / 255],
    "l2":   [0.25, 0.5, 1.0, 2.0],
    "l1":   [6.0, 12.0, 24.0],
}

device: cuda


In [34]:
def load_data(path):
    # The files store images as uint8. Convert them to floats in [0, 1] on the GPU.
    x, y = torch.load(path)
    return x.float().div(255).to(device), y.to(device)


# Training data: 10,000 images. Test data: the first 1,000 CIFAR-10 test images.
x_train, y_train = load_data("cifar10_train_10k.pt")
x_test, y_test = load_data("cifar10_test_1k.pt")
print("train:", tuple(x_train.shape), "test:", tuple(x_test.shape))


def augment(x):
    # Random horizontal flip and random crop (padding 4) for each image.
    n = x.shape[0]
    flip = torch.rand(n, device=x.device) < 0.5
    x = torch.where(flip.view(-1, 1, 1, 1), x.flip(3), x)
    padded = F.pad(x, (4, 4, 4, 4))
    i = torch.randint(0, 9, (n, 1, 1), device=x.device) + torch.arange(32, device=x.device).view(1, 32, 1)
    j = torch.randint(0, 9, (n, 1, 1), device=x.device) + torch.arange(32, device=x.device).view(1, 1, 32)
    return padded[torch.arange(n, device=x.device).view(n, 1, 1), :, i, j].permute(0, 3, 1, 2).contiguous()


def batches(x, y, batch_size, shuffle):
    idx = torch.randperm(len(x), device=x.device) if shuffle else torch.arange(len(x), device=x.device)
    for k in range(0, len(x), batch_size):
        j = idx[k:k + batch_size]
        yield x[j], y[j]

train: (10000, 3, 32, 32) test: (1000, 3, 32, 32)


In [35]:
# PreActResNet-18 with half width (2.8M parameters). The input is an image in [0, 1].
class PreActBlock(nn.Module):
    def __init__(self, in_planes, planes, stride):
        super().__init__()
        self.bn1 = nn.BatchNorm2d(in_planes)
        self.conv1 = nn.Conv2d(in_planes, planes, 3, stride, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(planes, planes, 3, 1, 1, bias=False)
        self.shortcut = None
        if stride != 1 or in_planes != planes:
            self.shortcut = nn.Conv2d(in_planes, planes, 1, stride, bias=False)

    def forward(self, x):
        out = F.relu(self.bn1(x))
        shortcut = self.shortcut(out) if self.shortcut is not None else x
        out = self.conv1(out)
        out = self.conv2(F.relu(self.bn2(out)))
        return out + shortcut


class PreActResNet18(nn.Module):
    def __init__(self, width=32, num_classes=10):
        super().__init__()
        self.register_buffer("mean", torch.tensor([0.4914, 0.4822, 0.4465]).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor([0.2471, 0.2435, 0.2616]).view(1, 3, 1, 1))
        self.conv1 = nn.Conv2d(3, width, 3, 1, 1, bias=False)
        layers, in_planes = [], width
        for mult, stride in [(1, 1), (1, 1), (2, 2), (2, 1), (4, 2), (4, 1), (8, 2), (8, 1)]:
            layers.append(PreActBlock(in_planes, mult * width, stride))
            in_planes = mult * width
        self.layers = nn.Sequential(*layers)
        self.bn = nn.BatchNorm2d(in_planes)
        self.linear = nn.Linear(in_planes, num_classes)

    def forward(self, x):
        out = self.layers(self.conv1((x - self.mean) / self.std))
        out = F.adaptive_avg_pool2d(F.relu(self.bn(out)), 1).flatten(1)
        return self.linear(out)


def make_model():
    return PreActResNet18().to(device)

# Implement the Attacks

Each attack is untargeted, uses the cross-entropy loss, and returns `x_adv` with
$\|x_{adv} - x\|_p \leq \epsilon$ and $x_{adv} \in [0, 1]$. The attacks must not change the mode (`train` or `eval`) of the model.

* `pgd_linf`: start at a uniformly random point in the $\ell_\infty$ ball. For each step, add `alpha * sign(grad)`,
  clip the perturbation to $[-\epsilon, \epsilon]$, and clip the image to $[0, 1]$.
* `pgd_l2`: start at a random point in the $\ell_2$ ball. For each step, add `alpha * grad / ||grad||_2`,
  scale the perturbation back to the $\ell_2$ ball if its norm is larger than $\epsilon$, and clip the image to $[0, 1]$.

You can use your code from HW1. The $\ell_1$ attack (sparse $\ell_1$ descent, Tramer & Boneh 2019) is complete.

In [36]:
def clip_box(x, delta):
    # Make x + delta an image in [0, 1]. This never increases |delta_i|.
    return (x + delta).clamp(0, 1) - x


def input_grad(model, x, delta, y):
    delta = delta.detach().requires_grad_(True)
    loss = F.cross_entropy(model(x + delta), y, reduction="sum")
    (grad,) = torch.autograd.grad(loss, delta)
    return grad.detach()


def l2_norm(v):
    return v.flatten(1).norm(dim=1).view(-1, 1, 1, 1)


def random_start(x, eps, attack="pgd_linf"):
    # Returns a perturbation delta with ||delta|| <= eps and x + delta in [0, 1].
    if attack == "pgd_linf":
        delta = torch.empty_like(x).uniform_(-eps, eps)
    elif attack == "pgd_l2":
        delta = torch.empty_like(x).normal_()
        n = l2_norm(delta)
        r = torch.zeros_like(n).uniform_(0, 1)
        delta = delta * (r / n * eps)
    else:
        raise ValueError(f"unknown attack: {attack}")
    return clip_box(x, delta).detach()


def pgd_linf(model, x, y, eps, alpha, steps, rand_init=True):
    x = x.detach()
    delta = random_start(x, eps, "pgd_linf") if rand_init else torch.zeros_like(x)

    for _ in range(steps):
        grad = input_grad(model, x, delta, y)
        delta = (delta + alpha * grad.sign()).clamp(-eps, eps)  # L-inf projection
        delta = clip_box(x, delta)                              # image-domain clip

    return (x + delta).detach()


def pgd_l2(model, x, y, eps, alpha, steps, rand_init=True):
    x = x.detach()
    tiny = 1e-10
    delta = random_start(x, eps, "pgd_l2") if rand_init else torch.zeros_like(x)

    for _ in range(steps):
        grad = input_grad(model, x, delta, y)
        grad = grad / (l2_norm(grad) + tiny)                    # normalized ascent direction
        delta = delta + alpha * grad
        delta = delta * (eps / (l2_norm(delta) + tiny)).clamp(max=1.0)  # L2 ball projection
        delta = clip_box(x, delta)                              # image-domain clip

    return (x + delta).detach()

In [37]:
def project_l1(delta, eps):
    # Projection of each image of delta onto the L1 ball of radius eps (Duchi et al., 2008).
    v = delta.flatten(1)
    a = v.abs()
    outside = a.sum(dim=1) > eps
    if not outside.any():
        return delta
    mu, _ = a.sort(dim=1, descending=True)
    cumsum = mu.cumsum(dim=1)
    idx = torch.arange(1, v.shape[1] + 1, device=v.device, dtype=v.dtype)
    rho = ((mu - (cumsum - eps) / idx > 0) * idx).argmax(dim=1, keepdim=True)
    theta = (cumsum.gather(1, rho) - eps) / (rho + 1).to(v.dtype)
    w = (a - theta).clamp_min(0) * v.sign()
    return torch.where(outside.unsqueeze(1), w, v).view(delta.shape)


def pgd_l1(model, x, y, eps, alpha, steps, rand_init=True, k=20):
    # Each step changes only the k coordinates with the largest gradient.
    # Coordinates at the image boundary that point outward are skipped.
    delta = torch.zeros_like(x)
    if rand_init:
        delta = torch.distributions.Laplace(0.0, 1.0).sample(x.shape).to(x.device)
        r = torch.rand(x.shape[0], 1, 1, 1, device=x.device)
        delta = delta / delta.flatten(1).abs().sum(1).view(-1, 1, 1, 1) * eps * r
    delta = clip_box(x, delta)
    for _ in range(steps):
        grad = input_grad(model, x, delta, y).flatten(1)
        x_cur = (x + delta).flatten(1)
        blocked = ((grad > 0) & (x_cur >= 1)) | ((grad < 0) & (x_cur <= 0))
        grad = grad.masked_fill(blocked, 0)
        top = grad.abs().topk(k, dim=1).indices
        step = torch.zeros_like(grad).scatter(1, top, grad.gather(1, top).sign())
        delta = clip_box(x, delta + alpha * (step / k).view_as(x))
        delta = project_l1(delta, eps)
    return (x + delta).detach()


ATTACKS = {"linf": pgd_linf, "l2": pgd_l2, "l1": pgd_l1}


def attack(model, x, y, norm, steps=10, eps=None, rand_init=True):
    # eps is used when attack is performed for epsilon
    # other than the default for that norm. (for eval)
    eps = EPS[norm] if eps is None else eps
    alpha = eps / steps ** 0.5 if norm == "l1" else 2.5 * eps / steps
    return ATTACKS[norm](model, x, y, eps, alpha, steps, rand_init)

# Evaluate Single and Multi-Norm Robust Accuracy

`test_model` gives the clean accuracy, the robust accuracy for each norm, and the union accuracy (in %) on the
1,000 test images. An image is robust for a norm if the model is correct on the clean image and on the result
of each restart of the attack. An image is robust for the union if it is robust for all three norms simultaneously.
Use the default settings (`steps=50, restarts=2`) for all the numbers in your report.

In [38]:
def test_model(model, steps=50, restarts=2, batch_size=250, eps=None, verbose=True, rand_init=True):
    eps = eps or EPS  # dict: norm -> epsilon; default = all three norms at their training eps
    use_union = len(eps) > 1  # union is only meaningful with more than one norm
    model.eval()
    torch.manual_seed(0)  # the same random starts of the attacks for each model, so the results are comparable
    start = time.time()
    correct = {k: 0 for k in ("clean", *eps, "union")}
    for i in range(0, len(x_test), batch_size):
        x, y = x_test[i:i + batch_size], y_test[i:i + batch_size]
        with torch.no_grad():
            clean = model(x).argmax(1) == y
        union = clean.clone()
        for norm, e in eps.items():
            robust = clean.clone()
            for _ in range(restarts):
                x_adv = attack(model, x, y, norm, steps, e, rand_init)
                with torch.no_grad():
                    robust &= model(x_adv).argmax(1) == y
            correct[norm] += robust.sum().item()
            if use_union:
                union &= robust
        correct["clean"] += clean.sum().item()
        if use_union:
          correct["union"] += union.sum().item()
    acc = {k: 100.0 * v / len(y_test) for k, v in correct.items()}
    if verbose:
        print("  ".join(f"{k} {v:.1f}" for k, v in acc.items()) + f"   ({time.time() - start:.0f} s)")
    return acc

# Adversarial Training

`train(model, loss_fn, ...)` fine-tunes `model` for `epochs` epochs on the training subset.
For each batch, it calls `loss_fn(model, x, y, **kwargs)` and does one optimizer step on the loss.

A loss function makes the adversarial examples itself. Put the model in `eval` mode for the attack
(so that the attack does not change the BatchNorm statistics), and in `train` mode before you calculate the loss.

In [39]:
def train(model, loss_fn, epochs=10, lr=0.01, batch_size=128, seed=0, **kwargs):
    torch.manual_seed(seed)
    opt = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=5e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, epochs * (len(x_train) // batch_size))
    params = list(model.parameters())
    start = time.time()
    for epoch in range(epochs):
        losses = []
        for x, y in batches(x_train, y_train, batch_size, shuffle=True):
            if len(y) < batch_size:  # skip the last, smaller batch
                continue
            x = augment(x)
            loss = loss_fn(model, x, y, **kwargs)
            opt.zero_grad()
            loss.backward()
            opt.step()
            sched.step()
            losses.append(loss.item())
        print(f"epoch {epoch + 1}: loss {np.mean(losses):.3f} ({time.time() - start:.0f} s)")
    model.eval()
    return model

# Part (a): Single-Norm Adversarial Training

Complete `single_norm_loss`: attack the batch with the norm `norm` (use `attack`), and return the
cross-entropy loss of the model on the adversarial images.

In [40]:
def single_norm_loss(model, x, y, norm):
    # put the model in eval mode, attack with `norm`, put the model in train mode, and return the loss
    model.eval()
    adv_x = attack(model, x, y, norm)
    model.train()
    return F.cross_entropy(model(adv_x), y)

def cross_entropy_loss(model, x, y):
    return F.cross_entropy(model(x), y)

In [41]:
model_standard = train(make_model(), cross_entropy_loss, epochs=5)
# test with EPS = {"linf": 8 / 255, "l2": 0.5, "l1": 12.0}, to get union as well.
print("\nStandard Model PGD Results:")
test_model(model_standard)

epoch 1: loss 1.876 (5 s)
epoch 2: loss 1.538 (10 s)
epoch 3: loss 1.380 (15 s)
epoch 4: loss 1.229 (20 s)
epoch 5: loss 1.139 (25 s)

Standard Model PGD Results:
clean 57.8  linf 0.2  l2 8.8  l1 1.6  union 0.2   (122 s)


{'clean': 57.8, 'linf': 0.2, 'l2': 8.8, 'l1': 1.6, 'union': 0.2}

In [42]:
model_linf = train(make_model(), single_norm_loss, epochs=5, norm="linf")
# test with EPS = {"linf": 8 / 255, "l2": 0.5, "l1": 12.0}, to get union as well.
print("\nl-inf adversarially trained Model PGD Results:")
test_model(model_linf)

epoch 1: loss 2.214 (42 s)
epoch 2: loss 2.100 (83 s)
epoch 3: loss 2.047 (126 s)
epoch 4: loss 2.027 (168 s)
epoch 5: loss 2.011 (210 s)

l-inf adversarially trained Model PGD Results:
clean 36.7  linf 23.1  l2 28.6  l1 20.5  union 18.8   (128 s)


{'clean': 36.7, 'linf': 23.1, 'l2': 28.6, 'l1': 20.5, 'union': 18.8}

In [43]:
# test with EPS = {"linf": 8 / 255, "l2": 0.5, "l1": 12.0}, to get union as well.
# rand_init=False for FGSM.
print("\nStandard Model FGSM Results:")
test_model(model_standard, steps=1, rand_init=False)
print("\nl-inf adversarially trained Model FGSM Results:")
test_model(model_linf, steps=1, rand_init=False)


Standard Model FGSM Results:
clean 57.8  linf 2.6  l2 16.0  l1 32.6  union 2.6   (4 s)

l-inf adversarially trained Model FGSM Results:
clean 36.7  linf 23.9  l2 29.0  l1 26.4  union 22.5   (4 s)


{'clean': 36.7, 'linf': 23.9, 'l2': 29.0, 'l1': 26.4, 'union': 22.5}

In [47]:
# test with EPS_SWEEP = {
#     "linf": [1 / 255, 2 / 255, 4 / 255, 8 / 255, 16 / 255],
#     "l2":   [0.25, 0.5, 1.0, 2.0],
#     "l1":   [6.0, 12.0, 24.0],
# }
# Here union is not valid, since we have multiple eps, passing them one by one.

models = {"standard": model_standard, "linf-AT": model_linf}
sweep = {}  # (model, steps, norm) -> robust accs, one per eps in EPS_SWEEP[norm]

for name, m in models.items():
    for steps in (1, 50):                      # 1 = FGSM, 50 = PGD
        for norm in NORMS:
            print(name, steps, norm)
            sweep[(name, steps, norm)] = [
                # rand_init=False for FGSM.
                test_model(m, steps=steps, eps={norm: e}, rand_init=(steps > 1))[norm]
                for e in EPS_SWEEP[norm]
            ]

standard 1 linf
clean 57.8  linf 39.4  union 0.0   (1 s)
clean 57.8  linf 26.7  union 0.0   (1 s)
clean 57.8  linf 12.6  union 0.0   (1 s)
clean 57.8  linf 2.6  union 0.0   (1 s)
clean 57.8  linf 0.5  union 0.0   (1 s)
standard 1 l2
clean 57.8  l2 30.3  union 0.0   (1 s)
clean 57.8  l2 16.0  union 0.0   (1 s)
clean 57.8  l2 4.8  union 0.0   (1 s)
clean 57.8  l2 1.0  union 0.0   (1 s)
standard 1 l1
clean 57.8  l1 33.8  union 0.0   (1 s)
clean 57.8  l1 32.6  union 0.0   (1 s)
clean 57.8  l1 33.5  union 0.0   (1 s)
standard 50 linf
clean 57.8  linf 37.5  union 0.0   (40 s)
clean 57.8  linf 21.9  union 0.0   (41 s)
clean 57.8  linf 4.8  union 0.0   (44 s)
clean 57.8  linf 0.2  union 0.0   (42 s)
clean 57.8  linf 0.0  union 0.0   (43 s)
standard 50 l2
clean 57.8  l2 26.4  union 0.0   (43 s)
clean 57.8  l2 8.8  union 0.0   (43 s)
clean 57.8  l2 0.5  union 0.0   (43 s)
clean 57.8  l2 0.0  union 0.0   (43 s)
standard 50 l1
clean 57.8  l1 11.3  union 0.0   (43 s)
clean 57.8  l1 1.5  union 0.0  

In [48]:
for norm in NORMS:
    print(f"\n{norm}  eps = {[round(e, 4) for e in EPS_SWEEP[norm]]}")
    for (name, steps, n), accs in sweep.items():
        if n == norm:
            label = "FGSM" if steps == 1 else "PGD-50"
            print(f"  {name:9s} {label:7s}", "  ".join(f"{a:5.1f}" for a in accs))


linf  eps = [0.0039, 0.0078, 0.0157, 0.0314, 0.0627]
  standard  FGSM     39.4   26.7   12.6    2.6    0.5
  standard  PGD-50   37.5   21.9    4.8    0.2    0.0
  linf-AT   FGSM     34.6   32.8   28.9   23.9   16.4
  linf-AT   PGD-50   34.6   32.8   29.1   23.1   13.2

l2  eps = [0.25, 0.5, 1.0, 2.0]
  standard  FGSM     30.3   16.0    4.8    1.0
  standard  PGD-50   26.4    8.8    0.5    0.0
  linf-AT   FGSM     32.9   29.0   23.7   17.0
  linf-AT   PGD-50   32.9   28.6   22.5   12.1

l1  eps = [6.0, 12.0, 24.0]
  standard  FGSM     33.8   32.6   33.5
  standard  PGD-50   11.3    1.5    0.2
  linf-AT   FGSM     28.5   26.4   26.2
  linf-AT   PGD-50   26.6   20.4   11.9


## (a) Standard and ℓ∞ adversarially trained models

All numbers are on the 1,000 test images, using PGD with 50 steps and 2 restarts. Accuracy is in %.

**Default budgets** (ℓ∞: 8/255, ℓ2: 0.5, ℓ1: 12):

| Model | Clean | ℓ∞ PGD | ℓ2 PGD | ℓ1 PGD | Union |
|---|---|---|---|---|---|
| Standard | 57.8 | 0.2 | 8.8 | 1.6 | 0.2 |
| ℓ∞-AT | 36.7 | 23.1 | 28.6 | 20.5 | 18.8 |

**PGD-50 at different ε.** Clean accuracy is 57.8 (standard) and 36.7 (ℓ∞-AT).

| Norm | Model | ε |  |  |  |  |
|---|---|---|---|---|---|---|
| ℓ∞ | | 1/255 | 2/255 | 4/255 | 8/255 | 16/255 |
| ℓ∞ | Standard | 37.5 | 21.9 | 4.8 | 0.2 | 0.0 |
| ℓ∞ | ℓ∞-AT | 34.6 | 32.8 | 29.1 | 23.1 | 13.2 |
| ℓ2 | | 0.25 | 0.5 | 1.0 | 2.0 | |
| ℓ2 | Standard | 26.4 | 8.8 | 0.5 | 0.0 | |
| ℓ2 | ℓ∞-AT | 32.9 | 28.6 | 22.5 | 12.1 | |
| ℓ1 | | 6 | 12 | 24 | | |
| ℓ1 | Standard | 11.3 | 1.5 | 0.2 | | |
| ℓ1 | ℓ∞-AT | 26.6 | 20.4 | 11.9 | | |

**Observations.**
- The standard model has the best clean accuracy (57.8%) but no robustness: 0.2% under ℓ∞ PGD at 8/255, and already 4.8% at 4/255.
- Adversarial training gives 23.1% robust accuracy at 8/255, but the clean accuracy drops from 57.8% to 36.7%. This is the robustness–accuracy trade-off.
- Robustness transfers to ℓ2 and ℓ1 even though training used only ℓ∞ (ℓ2: 28.6 vs. 8.8, ℓ1: 20.5 vs. 1.6).
- The adversarially trained model is not better at every ε: at 1/255 the standard model scores higher (37.5 vs. 34.6) because of its higher clean accuracy.
- The absolute numbers are low (5 epochs, 10k images, small network, the adversarial training loss is still about 2.0).

## (b) PGD vs. FGSM and different ε

FGSM is one attack step with no random start (`steps=1, rand_init=False`). PGD is 50 steps with 2 restarts. Accuracy under the ℓ∞ attack:

| ε | 1/255 | 2/255 | 4/255 | 8/255 | 16/255 |
|---|---|---|---|---|---|
| Standard, FGSM | 39.4 | 26.7 | 12.6 | 2.6 | 0.5 |
| Standard, PGD | 37.5 | 21.9 | 4.8 | 0.2 | 0.0 |
| ℓ∞-AT, FGSM | 34.6 | 32.8 | 28.9 | 23.9 | 16.4 |
| ℓ∞-AT, PGD | 34.6 | 32.8 | 29.1 | 23.1 | 13.2 |

Accuracy under the ℓ2 attack and under the ℓ1 attack:

| | ℓ2, ε = 0.25 | 0.5 | 1.0 | 2.0 | ℓ1, ε = 6 | 12 | 24 |
|---|---|---|---|---|---|---|---|
| Standard, FGSM | 30.3 | 16.0 | 4.8 | 1.0 | 33.8 | 32.6 | 33.5 |
| Standard, PGD | 26.4 | 8.8 | 0.5 | 0.0 | 11.3 | 1.5 | 0.2 |
| ℓ∞-AT, FGSM | 32.9 | 29.0 | 23.7 | 17.0 | 28.5 | 26.4 | 26.2 |
| ℓ∞-AT, PGD | 32.9 | 28.6 | 22.5 | 12.1 | 26.6 | 20.4 | 11.9 |

**1. PGD on the adversarially trained model vs. the standard model.** In the ℓ∞ table, the PGD accuracy of the adversarially trained model stays well above that of the standard model once ε is 2/255 or larger (32.8% vs. 21.9% at 2/255, 29.1% vs. 4.8% at 4/255, 23.1% vs. 0.2% at 8/255, 13.2% vs. 0.0% at 16/255). The same holds for ℓ2 and ℓ1. At 1/255 the standard model is better (37.5% vs. 34.6%), because of its higher clean accuracy (57.8% vs. 36.7%). So the adversarially trained model can still be attacked, but it is much harder to attack.

**2. FGSM against both models.**
- ℓ∞, standard model: at 8/255, FGSM alone already brings the accuracy down to 2.6%, and PGD furthers it to 0.2%.
- ℓ∞, adversarially trained model: at 8/255, FGSM (23.9%) is almost as strong as PGD (23.1%).
- ℓ∞, adversarially trained model: the gap between FGSM and PGD grows with ε. It is zero at 1/255 and 2/255, and at 16/255 FGSM gives 16.4% and PGD 13.2%.
- ℓ2 behaves like ℓ∞: PGD is slightly stronger than FGSM, and the gap on the adversarially trained model grows with ε (at 1.0 FGSM gives 23.7% and PGD 22.5%, at 2.0 FGSM gives 17.0% and PGD 12.1%).
- ℓ1: FGSM changes the accuracy of the standard model very little (about 33% at all three ε), while PGD brings it down to 0.2% at ε = 24. On the adversarially trained model, FGSM gives 26.2% and PGD 11.9% at ε = 24.

**3. Different ε against the adversarially trained model (trained at 8/255).**
- The accuracy falls gradually as ε grows: 34.6%, 32.8%, 29.1%, 23.1% and 13.2% at 1/255, 2/255, 4/255, 8/255 and 16/255 (PGD), compared to 36.7% clean, with no sudden drop. The robustness is tied to the trained budget and degrades gradually beyond it.
- For comparison, the standard model falls faster: 37.5%, 21.9%, 4.8%, 0.2% and 0.0% at the same ε.
- ℓ2 and ℓ1 show the same gradual decline with ε under PGD (tables above).